# What Does the Model Pay Attention To?
### Feature Exploration — Clinical Trial Prompts

This notebook walks through a clinical trial prompt and shows **which internal concepts
(features) the model activates**, and **how much each one contributes** to the model's
prediction.

No machine learning background needed to read this. Each feature has a plain-English label.

---
**Before running:** set `CHECKPOINT_PATH` and `PROMPT_ID` below.

In [ ]:
import json
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

from clt.config import CLTConfig
from clt.model import CrossLayerTranscoder
from viz.features import plot_top_features, plot_activation_heatmap

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# --- Configure these ---
CHECKPOINT_PATH = "checkpoints/pythia-410m-4096/clt_final.pt"
FEATURE_LABELS_PATH = "data/feature_labels.jsonl"
PROMPT_FILE = "prompts/eligibility.py"   # or adverse_events.py, endpoints.py
PROMPT_ID = "eligible_inclusion"          # TrialPrompt id to run

LAYER_TO_INSPECT = 12   # middle layer — good starting point
TOPK_FEATURES = 15

# Pythia-410m defaults
CFG = CLTConfig(
    n_layers=24,
    d_model=1024,
    d_mlp=4096,
    n_features=2048,
)

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

## Load model + CLT

In [ ]:
pythia = HookedTransformer.from_pretrained("EleutherAI/pythia-410m").to(device)
pythia.eval()

clt = CrossLayerTranscoder(CFG).to(device)
clt.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True))
clt.eval()

# Load feature labels
feature_labels: dict[str, str] = {}
try:
    with open(FEATURE_LABELS_PATH) as f:
        for line in f:
            obj = json.loads(line)
            feature_labels[str(obj["feature_index"])] = obj["label"]
    print(f"Loaded {len(feature_labels)} feature labels.")
except FileNotFoundError:
    print("No feature_labels.jsonl yet — features will show as 'Feature N'.")

## Load and display the prompt

In [ ]:
# Import prompts from the configured file
import importlib.util, sys
spec = importlib.util.spec_from_file_location("prompt_module", PROMPT_FILE)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

# Each prompts/*.py exports ELIGIBILITY_PROMPTS, ADVERSE_EVENT_PROMPTS, or ENDPOINT_PROMPTS
# Fall back to checking all exported lists
all_prompts = []
for attr in dir(mod):
    val = getattr(mod, attr)
    if isinstance(val, list) and val and isinstance(val[0], dict) and 'id' in val[0]:
        all_prompts.extend(val)

prompt = next(p for p in all_prompts if p['id'] == PROMPT_ID)

print(f"Prompt ID:     {prompt['id']}")
print(f"Domain tags:   {', '.join(prompt['domain_tags'])}")
print(f"Target token:  '{prompt['target_token']}'")
print()
print(prompt['prompt'])

## Extract residual streams + run CLT

In [ ]:
tokens = pythia.to_tokens(prompt['prompt'])  # (1, seq)
token_strings = pythia.to_str_tokens(prompt['prompt'])

resid_streams: list[torch.Tensor] = []

with torch.no_grad():
    _, cache = pythia.run_with_cache(tokens)
    for l in range(CFG.n_layers):
        # (1, seq, d_model)
        resid_streams.append(cache[f'blocks.{l}.hook_resid_pre'])

with torch.no_grad():
    feature_acts = clt.encode(resid_streams)

print(f"Sequence length: {tokens.shape[1]} tokens")
print(f"Tokens: {token_strings}")

## Which features are most active?

The chart below shows the strongest internal concepts at a key token position.
Each bar represents one learned feature — longer bars mean the model is
relying more heavily on that concept.

In [ ]:
# Inspect the last token position (where the model makes its prediction)
target_position = tokens.shape[1] - 1

fig = plot_top_features(
    feature_acts,
    layer=LAYER_TO_INSPECT,
    position=target_position,
    topk=TOPK_FEATURES,
    token_strings=token_strings,
    feature_labels=feature_labels,
)
plt.show()

## Where in the prompt do these concepts fire?

The heatmap below shows which tokens (rows) activate which features (columns).
Dark blue = strong activation. This reveals *which parts of the text*
the model is focusing on.

In [ ]:
fig = plot_activation_heatmap(
    feature_acts,
    token_strings=token_strings,
    layer=LAYER_TO_INSPECT,
    topk_features=TOPK_FEATURES,
    feature_labels=feature_labels,
)
plt.show()

## Active feature summary table

All features that fired at the final token position, sorted by activation strength.

In [ ]:
import pandas as pd

active = clt.active_features(feature_acts, threshold=1e-6)
active_at_target = [
    r for r in active
    if r["layer"] == LAYER_TO_INSPECT and r["pos"] == target_position
]
active_at_target.sort(key=lambda r: r["activation"], reverse=True)

rows = []
for r in active_at_target:
    rows.append({
        "Feature": feature_labels.get(str(r["feature"]), f"Feature {r['feature']}"),
        "Index": r["feature"],
        "Activation": f"{r['activation']:.4f}",
    })

pd.DataFrame(rows)